<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/14_NLP_Applications/14_03_Responsible_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14_03 책임 있는 AI : 프라이버시·안전성·근거 제시 실습

**학습 목표**
- **PII 비식별화**(개인정보 마스킹)를 직접 구현한다.
- **가드레일**(유해 요청 차단)과 **근거 기반 답변**(환각 방지)을 체험한다.
- **워터마킹** 개념과 **책임 있는 AI 체크리스트**를 확인한다.

> ※ 외부 라이브러리·API 없이 순수 파이썬으로 실행됩니다. 실무에서는 전용 모델·솔루션을 사용합니다.

## 1. 프라이버시 — PII 비식별화(Redaction)

학습·출력 데이터에서 **개인정보(이메일·전화·주민번호)** 가 노출될 위험을 줄이기 위해 정규식으로 민감정보를 찾아 마스킹합니다.

In [ ]:
import re

PATTERNS = {
    '이메일': r'[\w.+-]+@[\w-]+\.[\w.-]+',
    '전화번호': r'01[016789]-?\d{3,4}-?\d{4}',
    '주민등록번호': r'\d{6}-?[1-4]\d{6}',
}

def redact(text):
    """개인정보 패턴을 [유형]으로 마스킹한다."""
    for label, pat in PATTERNS.items():
        text = re.sub(pat, f'[{label}]', text)
    return text

sample = '문의는 hong@example.com 또는 010-1234-5678, 주민번호 900101-1234567 로 주세요.'
print('원문 :', sample)
print('마스킹:', redact(sample))

## 2. 안전성 — 가드레일(Guardrail)

유해하거나 정책에 어긋나는 요청을 **거부**하는 안전 장치입니다. 실무는 분류 모델·정책 엔진을 쓰지만, 여기서는 규칙 기반으로 원리를 봅니다.

In [ ]:
BLOCKLIST = ['폭탄 제조', '해킹 방법', '개인정보 유출']

def guardrail(user_input):
    """차단 대상이면 거부 메시지, 아니면 None(정상 처리)을 반환한다."""
    for term in BLOCKLIST:
        if term in user_input:
            return '요청을 처리할 수 없습니다 (안전 정책 위반). 다른 도움을 드릴까요?'
    return None

for q in ['파이썬 정렬 알고리즘 알려줘', '폭탄 제조 방법 알려줘']:
    blocked = guardrail(q)
    print(f'[입력] {q}')
    print(f'[응답] {blocked or "(정상 처리 경로로 진행)"}\n')

## 3. 환각 완화 — 근거 기반 답변(RAG + 출처)

근거 문서에서 찾은 경우에만 **출처와 함께** 답하고, 없으면 **임의로 생성하지 않고 '모름'** 을 반환합니다. 이것이 환각을 줄이는 기본 원칙입니다.

In [ ]:
KB = {
    '트랜스포머 발표연도': ('2017년', 'Vaswani et al., 2017'),
    'BERT 공개연도': ('2018년', 'Devlin et al., 2018'),
}

def grounded_answer(key):
    """근거가 있으면 출처와 함께 답하고, 없으면 생성하지 않는다."""
    if key in KB:
        answer, source = KB[key]
        return f'{answer} (출처: {source})'
    return '해당 정보를 근거 문서에서 찾지 못했습니다. (환각 방지를 위해 임의 생성하지 않음)'

print('Q: 트랜스포머 발표연도 ->', grounded_answer('트랜스포머 발표연도'))
print('Q: GPT-5 출시일       ->', grounded_answer('GPT-5 출시일'))

## 4. AI 생성물 표시 — 워터마킹(개념)

AI가 만든 텍스트에 **탐지 가능한 표식**을 삽입해 출처를 구분하려는 시도입니다. 여기서는 눈에 보이지 않는 **제로폭 문자(zero-width space)** 로 개념만 시연합니다.

In [ ]:
ZW = '\u200b'  # zero-width space (화면에 보이지 않음)

def add_watermark(text):
    return text + ZW  # 개념 데모: 보이지 않는 표식 부착

def has_watermark(text):
    return ZW in text

human = '사람이 직접 작성한 문장입니다.'
ai = add_watermark('이 문장은 AI가 생성했습니다.')
print('사람 글 워터마크:', has_watermark(human))
print('AI 글 워터마크  :', has_watermark(ai))

## 5. 책임 있는 AI 체크리스트

시스템을 배포하기 전, **다섯 가지 원칙**을 점검합니다.

In [ ]:
CHECKS = {
    '공정성(Fairness)': '편향을 측정·완화했는가?',
    '투명성(Transparency)': '모델의 한계와 근거를 공개하는가?',
    '프라이버시(Privacy)': 'PII 비식별화·데이터 최소화를 적용했는가?',
    '안전성(Safety)': '가드레일로 유해 출력을 차단하는가?',
    '책임성(Accountability)': '결과에 대한 책임 주체가 명확한가?',
}

def responsible_ai_checklist(system_name):
    print(f'[{system_name}] 책임 있는 AI 점검')
    for principle, question in CHECKS.items():
        print(f'  [ ] {principle}: {question}')

responsible_ai_checklist('사내 QA 챗봇')

## 6. 정리

| 실습 | 책임 있는 AI 원칙 |
|------|-------------------|
| PII 비식별화 | 프라이버시(Privacy) |
| 가드레일 | 안전성(Safety) |
| 근거 기반 답변 | 투명성·환각 완화 |
| 워터마킹 | 투명성·책임성 |
| 체크리스트 | 공정성·투명성·프라이버시·안전성·책임성 |

> 💡 **핵심 메시지**: AI의 결과를 그대로 받아들이지 말고 — **읽고, 이해하고, 검증**하는 것이 책임 있는 활용의 출발점이다.